In [1]:
import pandas as pd


df = pd.read_excel('log_B.xlsx')

# df['del_G_eV'] = df['del G (eV) Paul'].fillna(df['del G (eV) Azida']).fillna(df['del G (eV) NBS'])
df['del_G_eV'] = df['del G (eV) Paul'].fillna(df['del G (eV) Azida']).fillna(df['del G (eV) Other'])

df = df[~df['ligand'].str.contains('NH4', na=False)]
df = df[~df['ligand'].str.contains('OH', na=False)]
df = df[~df['ligand'].str.contains('Cl', na=False)]
# df = df[~df['ligand'].str.contains('CN', na=False)]

new_df = df[['ligand', 'metal_ion', 'n_metal', 'n_complex', 'signed_metal_ion', 'del_G_eV','reference']]

new_df=new_df.dropna(how='any')
new_df


,ligand,metal_ion,n_metal,n_complex,signed_metal_ion,del_G_eV,reference
0,NH3,Ag+,1,1,Ag[1+],0.321990,Bjerrum1957StabilitySubstances
1,NH3,Ag+,1,2,Ag[1+],-0.190677,Bjerrum1957StabilitySubstances
2,NH3,Au+,1,2,Au[1+],-0.325391,Bjerrum1957StabilitySubstances
3,NH3,Au3+,1,4,Au[3+],1.681275,Bjerrum1957StabilitySubstances
4,NH3,Ca2+,1,1,Ca[2+],-6.001205,Bjerrum1957StabilitySubstances
...,...,...,...,...,...,...,...
141,CN[1-],Zn2+,1,4,Zn[2+],4.633763,Smith1989CriticalConstants
142,CN[1-],Pt2+,1,4,Pt[2+],5.646346,Bard2017StandardSolution
144,NO2[1-],Cu2+,1,1,Cu[2+],0.274026,Smith1989CriticalConstants
145,NO2[1-],Cu2+,1,2,Cu[2+],-0.072720,Smith1989CriticalConstants


In [2]:
new_df

def extract_ion_number(expression):
    if "[" in expression and "]" in expression:
        ion = expression[expression.find("[") + 1 : expression.find("]")]
        return int(ion.replace("+", "").replace("-", "")) * (-1 if "-" in ion else 1)
    return 0

# Function to format the charge
def format_charge(charge):
    if charge == 0:
        return ""
    sign = "+" if charge > 0 else "-"
    return f"{abs(charge)}{sign}"

# Create the new column "Species"
def create_species(row):
    # Extract charges
    metal_charge = extract_ion_number(row["signed_metal_ion"]) * row["n_metal"]
    ligand_charge = extract_ion_number(row["ligand"]) * row["n_complex"]
    total_charge = metal_charge + ligand_charge

    # Format total charge
    total_charge_str = format_charge(int(total_charge))

    # Format Species column
    metal_ion = row['signed_metal_ion'].split('[')[0]
    ligand = row['ligand'].split('[')[0]
    n_metal = str(int(row['n_metal'])).replace("1","")
    n_ligand = str(int(row['n_complex'])).replace("1","")
    
    if total_charge == 0:
        total_charge_super_script = ""
    else:
        total_charge_super_script = "^"
    
    species = f"[{metal_ion}{n_metal}({ligand}){n_ligand}]{total_charge_super_script}{total_charge_str}"
    return species.replace("_1","").replace("^1","").replace("+1","+").replace("-1","-")

new_df["Species"] = new_df.apply(create_species, axis=1)
new_df["Metal ion"] = new_df["signed_metal_ion"].str.replace("[", "^").str.replace("]","")

new_df


,ligand,metal_ion,n_metal,n_complex,signed_metal_ion,del_G_eV,reference,Species,Metal ion
0,NH3,Ag+,1,1,Ag[1+],0.321990,Bjerrum1957StabilitySubstances,[Ag(NH3)]+,Ag^1+
1,NH3,Ag+,1,2,Ag[1+],-0.190677,Bjerrum1957StabilitySubstances,[Ag(NH3)2]+,Ag^1+
2,NH3,Au+,1,2,Au[1+],-0.325391,Bjerrum1957StabilitySubstances,[Au(NH3)2]+,Au^1+
3,NH3,Au3+,1,4,Au[3+],1.681275,Bjerrum1957StabilitySubstances,[Au(NH3)4]^3+,Au^3+
4,NH3,Ca2+,1,1,Ca[2+],-6.001205,Bjerrum1957StabilitySubstances,[Ca(NH3)]^2+,Ca^2+
...,...,...,...,...,...,...,...,...,...
141,CN[1-],Zn2+,1,4,Zn[2+],4.633763,Smith1989CriticalConstants,[Zn(CN)4]^2-,Zn^2+
142,CN[1-],Pt2+,1,4,Pt[2+],5.646346,Bard2017StandardSolution,[Pt(CN)4]^2-,Pt^2+
144,NO2[1-],Cu2+,1,1,Cu[2+],0.274026,Smith1989CriticalConstants,[Cu(NO2)]+,Cu^2+
145,NO2[1-],Cu2+,1,2,Cu[2+],-0.072720,Smith1989CriticalConstants,[Cu(NO2)2],Cu^2+


In [3]:
def generate_ligand_complex_latex_table(df, ligand):
    # Filter the DataFrame for the specific ligand
    combined_data = []
    ligand_data = df[df['ligand'] == ligand]
    for _, row in ligand_data.iterrows():
        combined_data.append({
            'Species': row['Species'],
            'Metal ion': row['Metal ion'],
            'del_G_eV': row['del_G_eV'],
            'Reference': row['reference']
        })

    # Generate the LaTeX table with captions for continued pages
    if ligand == 'Gly[1-]':
        ligand_text = 'Gly-'
    if ligand == 'CN[1-]':
        ligand_text = 'CN-'
    if ligand == 'NH3':
        ligand_text = 'NH3'
    latex_table = fr"""\clearpage
\begin{{longtable}}{{|p{{4cm}}|p{{4cm}}|p{{3cm}}|p{{3cm}}|}}
\caption{{Formation energies of species for \ce{{{ligand_text}}} complexes.}} 
\label{{tab:{ligand}_complex_energies}}
\\
\hline
\textbf{{Species}} & \textbf{{Metal ion}} & \textbf{{\( \Delta G^\circ_{{298}} \) (eV)}} & \textbf{{Reference}} \\ \hline
\endfirsthead
\caption*{{Table \thetable\ continued from previous pages.}} \\
\hline
\textbf{{Species}} & \textbf{{Metal ion}} & \textbf{{\( \Delta G^\circ_{{298}} \) (eV)}} & \textbf{{Reference}} \\ \hline
\endhead
\hline
\endfoot
\hline
\endlastfoot
"""
    # Add rows to the table
    for i, entry in enumerate(combined_data):
        latex_table += f"\ce{{{entry['Species']}}} & \ce{{{entry['Metal ion']}}} & {entry['del_G_eV']:.3f} & \\textnormal{{\\citenum{{{entry['Reference']}}}}}"
        if i < len(combined_data) - 1:
            latex_table += " \\\\ \\hline\n"
#     for entry in combined_data:
#         latex_table += f"\ce{{{entry['Species']}}} & \ce{{{entry['Metal ion']}}} & {entry['del_G_eV']:.3f} & \\citenum{{{entry['Reference']}}} \\\\ \\hline\n"

    latex_table += r"\end{longtable}"

    # Save the LaTeX table to a file
    output_path = f'data/paper/ligand/{ligand}_complex_energy_table.tex'
    with open(output_path, 'w') as tex_file:
        tex_file.write(latex_table)
#     print(latex_table)
    print(f"LaTeX table saved to {output_path}")

# Example usage
generate_ligand_complex_latex_table(new_df, 'NH3')

generate_ligand_complex_latex_table(new_df, 'Gly[1-]')
generate_ligand_complex_latex_table(new_df, 'CN[1-]')
# 


LaTeX table saved to data/paper/ligand/NH3_complex_energy_table.tex
LaTeX table saved to data/paper/ligand/Gly[1-]_complex_energy_table.tex
LaTeX table saved to data/paper/ligand/CN[1-]_complex_energy_table.tex


In [4]:
import json 

def format_species(species):
#     print(species)
    species_text = species.split('(aq)')[0]
    total_charge = extract_ion_number(species_text)
    total_charge_str = format_charge(total_charge)
    if total_charge == 0:
        total_charge_super_script = ""
    else:
        total_charge_super_script = "^"
    species_text = f"{species_text.split('[')[0]}{total_charge_super_script}{total_charge_str}"
    return species_text.replace("_1","").replace("^1","").replace("+1","+").replace("-1","-")

def generate_bulk_metal_latex_table(metal):
    with open(f'data/{metal}_ion_formation_energy.json', 'r') as ion_file:
        ion_data = json.load(ion_file)
    with open(f'data/{metal}_solid_formation_energy.json', 'r') as solid_file:
        solid_data = json.load(solid_file)
        
    combined_data = []
    for species, energy in ion_data.items():
        species_name = format_species(species)
        combined_data.append({'Species': species_name, 'State': 'Aqueous ion', 'Energy': energy})
    for species, energy in solid_data.items():
        species_name = format_species(species)
        combined_data.append({'Species': species_name, 'State': 'Solid', 'Energy': energy})
    latex_table = fr"""\clearpage
\begin{{longtable}}{{|p{{4cm}}|p{{3cm}}|p{{3cm}}|}}
\caption{{Formation energies of {metal} species queried from Materials Project\cite{{Jain2013TheInnovation}}.}} 
\label{{tab:bulk_{metal}_energies}}
\\
\hline
\textbf{{Species}}  & \textbf{{State}} & \textbf{{\( \Delta G\) (eV)}} \\ \hline
\endfirsthead
\caption*{{Table \thetable\ continued from previous pages.}} \\
\hline
\textbf{{Species}}  & \textbf{{State}} & \textbf{{\( \Delta G\) (eV)}} \\ \hline
\endhead
\hline
\endfoot
\hline
\endlastfoot
"""
    
    for i, entry in enumerate(combined_data):
        latex_table += f"\ce{{{entry['Species']}}} & {entry['State']} & {entry['Energy']:.3f}"
        if i < len(combined_data) - 1:
            latex_table += " \\\\ \\hline\n"
    latex_table += r"\end{longtable}"
    output_path = f'data/paper/metal/{metal}_energy_table.tex'
    with open(output_path, 'w') as tex_file:
        tex_file.write(latex_table)
    print(latex_table)
    print(f"LaTeX table saved to {output_path}")
    
generate_bulk_metal_latex_table('Fe')



\clearpage
\begin{longtable}{|p{4cm}|p{3cm}|p{3cm}|}
\caption{Formation energies of Fe species queried from Materials Project\cite{Jain2013TheInnovation}.} 
\label{tab:bulk_Fe_energies}
\\
\hline
\textbf{Species}  & \textbf{State} & \textbf{\( \Delta G\) (eV)} \\ \hline
\endfirsthead
\caption*{Table \thetable\ continued from previous pages.} \\
\hline
\textbf{Species}  & \textbf{State} & \textbf{\( \Delta G\) (eV)} \\ \hline
\endhead
\hline
\endfoot
\hline
\endlastfoot
\ce{FeO2^2-} & Aqueous ion & -3.011 \\ \hline
\ce{FeOH+} & Aqueous ion & -2.824 \\ \hline
\ce{Fe(OH)3} & Aqueous ion & -6.784 \\ \hline
\ce{FeOH^2+} & Aqueous ion & -4.743 \\ \hline
\ce{FeO4^2-} & Aqueous ion & -3.290 \\ \hline
\ce{Fe^2+} & Aqueous ion & -0.768 \\ \hline
\ce{Fe^3+} & Aqueous ion & 0.002 \\ \hline
\ce{Fe(OH)2+} & Aqueous ion & -4.490 \\ \hline
\ce{FeO2-} & Aqueous ion & -3.767 \\ \hline
\ce{FeHO2-} & Aqueous ion & -3.866 \\ \hline
\ce{Fe100} & Solid & 27.946 \\ \hline
\ce{Fe28} & Solid & 4.908 \\ \hline
\

In [5]:
metal_list = ['Ti', 'Cu','Au','Pd','Pt']
for m in metal_list:
    
    generate_bulk_metal_latex_table(m)


\clearpage
\begin{longtable}{|p{4cm}|p{3cm}|p{3cm}|}
\caption{Formation energies of Ti species queried from Materials Project\cite{Jain2013TheInnovation}.} 
\label{tab:bulk_Ti_energies}
\\
\hline
\textbf{Species}  & \textbf{State} & \textbf{\( \Delta G\) (eV)} \\ \hline
\endfirsthead
\caption*{Table \thetable\ continued from previous pages.} \\
\hline
\textbf{Species}  & \textbf{State} & \textbf{\( \Delta G\) (eV)} \\ \hline
\endhead
\hline
\endfoot
\hline
\endlastfoot
\ce{Ti^3+} & Aqueous ion & -4.254 \\ \hline
\ce{TiO2^2+} & Aqueous ion & -5.461 \\ \hline
\ce{TiHO3-} & Aqueous ion & -10.483 \\ \hline
\ce{Ti^2+} & Aqueous ion & -3.889 \\ \hline
\ce{TiO^2+} & Aqueous ion & -6.593 \\ \hline
\ce{Ti100} & Solid & 13.833 \\ \hline
\ce{Ti6} & Solid & 1.862 \\ \hline
\ce{Ti3} & Solid & 0.000 \\ \hline
\ce{Ti} & Solid & 0.062 \\ \hline
\ce{Ti2} & Solid & 0.009 \\ \hline
\ce{TiH2} & Solid & -0.523 \\ \hline
\ce{Ti4H5} & Solid & -1.211 \\ \hline
\ce{TiH} & Solid & -0.172 \\ \hline
\ce{Ti2H4} & 